# GDrive Transfer with Rclone and Google Colab
This notebook sets up rclone, clones the repository, and runs the Web-UI to transfer files between cloud drives.

In [ ]:
# @title Install Dependencies & Setup Rclone
# @description Install rclone, cloudflared, clone the repository and change directory.
import os
import subprocess
import sys

# Clone repository if not already in the correct directory
if not os.path.exists("src/main.py"):
    print("Cloning repository...")
    # Run git clone and wait for it to complete
    result = subprocess.run("git clone https://github.com/Hikaner/GDrive-transfer.git", shell=True, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    
    if os.path.exists("GDrive-transfer"):
        os.chdir("GDrive-transfer")
        print(f"Changed directory to: {os.getcwd()}")
    else:
        print("Warning: GDrive-transfer directory not found after git clone. You might already be inside it or clone failed.")

print("Installing rclone...")
subprocess.run("curl https://rclone.org/install.sh | bash", shell=True)

print("Installing cloudflared...")
subprocess.run("curl -L --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared.deb && rm cloudflared.deb", shell=True)

print("Installing python dependencies...")
subprocess.run("pip install fastapi uvicorn python-multipart", shell=True)

print("Setup completed successfully!")

In [ ]:
# @title Run Web-UI & Expose with Cloudflared
# @description Start the FastAPI server and expose it via cloudflared tunnel.
import os
import subprocess
import time
import secrets
import threading
import sys
import re

# Ensure we are in the correct directory
if os.path.exists("GDrive-transfer") and not os.path.exists("src/main.py"):
    os.chdir("GDrive-transfer")

# Clean up any existing uvicorn or cloudflared processes from previous runs
print("Cleaning up existing processes...")
subprocess.run("pkill -f uvicorn", shell=True)
subprocess.run("pkill -f cloudflared", shell=True)
time.sleep(1)

# Generate a random token for authentication
AUTH_TOKEN = secrets.token_hex(16)

# Write the token to a file so main.py can read it or pass it via environment variable
os.environ["AUTH_TOKEN"] = AUTH_TOKEN

# Start FastAPI server in the background
print("Starting FastAPI server...")
# Bind to 0.0.0.0 instead of 127.0.0.1 to ensure cloudflared can route to it properly
backend_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "src.main:app", "--host", "0.0.0.0", "--port", "8000"],
    env=os.environ.copy()
)

# Wait for FastAPI to start
time.sleep(3)

# Start cloudflared tunnel in the background
print("Starting cloudflared tunnel...")
# Route to http://localhost:8000
cloudflared_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Function to read cloudflared output and extract the public URL
def monitor_tunnel():
    url_found = False
    for line in iter(cloudflared_process.stdout.readline, ''):
        # Look for the tunnel creation log line, avoiding request logs (which contain dest=, etc.)
        if "trycloudflare.com" in line and "dest=" not in line and "Request" not in line:
            # Use regex to find the exact URL starting with https:// and ending with trycloudflare.com
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                url = match.group(0)
                print("\n" + "="*80)
                print(f" GDrive-transfer Web-UI is live!")
                print(f" Access URL: {url}/?token={AUTH_TOKEN}")
                print("="*80 + "\n")
                url_found = True
                break # Stop monitoring once the URL is found and printed
        if url_found:
            break

# Start monitoring thread
threading.Thread(target=monitor_tunnel, daemon=True).start()

try:
    # Keep the cell running
    while True:
        if backend_process.poll() is not None:
            print("FastAPI server stopped.")
            break
        if cloudflared_process.poll() is not None:
            print("Cloudflared tunnel stopped.")
            break
        time.sleep(1)
except KeyboardInterrupt:
    print("Stopping services...")
    backend_process.terminate()
    cloudflared_process.terminate()
    print("Services stopped.")